# Small Object Detection and Tracking in UAV Imagery
YOLO fine-tuning on VisDrone, size-aware comparison of standard vs. sliced (SAHI) inference, ByteTrack tracking.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
!nvidia-smi

In [ ]:
GITHUB_USER = "mahdildarr"
!git clone https://github.com/{GITHUB_USER}/uav-small-object-detection.git
%cd uav-small-object-detection
!pip install -q -r requirements.txt

## 1. Training
VisDrone (~2 GB) is downloaded automatically on the first run. With `yolo11s` at 1024 px each epoch takes a few minutes on a T4; lower `--epochs` / `--imgsz` if time is limited.

In [ ]:
!python src/train.py --model yolo11s.pt --epochs 50 --imgsz 1024 --batch 8 --name yolo11s_1024

## 2. Standard vs. sliced inference by object size

In [ ]:
!python src/evaluate_small_objects.py --weights runs/visdrone/yolo11s_1024/weights/best.pt --limit 200 --save-vis 6

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("runs/eval/vis/*.jpg"))[:3]:
    display(Image(f, width=1200))

## 3. Tracking
Download the VisDrone2019-MOT-val sequences from the [official repository](https://github.com/VisDrone/VisDrone-Dataset) and update the paths below.

In [ ]:
SEQ = "/content/VisDrone2019-MOT-val/sequences/uav0000086_00000_v"
GT = "/content/VisDrone2019-MOT-val/annotations/uav0000086_00000_v.txt"
!python src/frames_to_video.py --frames {SEQ} --out sequence.mp4
!python src/track.py --weights runs/visdrone/yolo11s_1024/weights/best.pt --source sequence.mp4
!python src/eval_mot.py --weights runs/visdrone/yolo11s_1024/weights/best.pt --seq-dir {SEQ} --gt {GT}

## Save results
Copy the outputs to Google Drive and update the results table in the README with these numbers.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/uav-small-object-detection
!cp -r runs /content/drive/MyDrive/uav-small-object-detection/